**Two models to ensemble:**

waymo_rfdetr_150_best.pth — RF-DETR-L, mAP@0.5 = 0.826

waymo_yolo26x_150_best.pt — YOLO26x, mAP@0.5 = 0.779

**Goal:** Combine both predictions to exceed RF-DETR-L's 0.826 and ideally hit our production targets:

mAP@0.5 ≥ 0.80 ✅ (already hit)

Pedestrian recall ≥ 0.90 ← main gap

Cyclist recall ≥ 0.95 ← main gap

**Ensemble strategy:**

DISTILLATION PIPELINE
═══════════════════════════════════════════

STEP 1 — WBF (tool to generate better labels)
─────────────────────────────────────────────
```
Image → RF-DETR-L  → boxes ─┐
                             ├── WBF merge → pseudo-labels
Image → YOLO26x    → boxes ─┘
```
STEP 2 — DISTILLATION (train student on those labels)
───────────────────────────────────────────────

Image + pseudo-labels → Train YOLOv8m student → fast model

**How WBF works:**

WBF (Weighted Box Fusion):an algorithm to merge bounding boxes from multiple models into one final box.

1. Collect all boxes from both models
2. Group boxes that overlap (same object)
3. For each group:
   - Average the coordinates (weighted by confidence)
   - Boost the confidence score
4. Output one clean box per object

WBF is a tool inside distillation — it creates the high quality pseudo-labels that the student model learns from.

Distillation (train a new model on ensemble outputs)

Distillation = two smart teachers label everything perfectly → one fast student learns from those perfect labels → student nearly matches teachers but runs 6x faster

**Simple analogy:**

WBF = two teachers GRADING a test together
      (Teacher 1 says box here, Teacher 2 says box there,
       WBF combines → best possible answer)

Distillation = student STUDYING from those graded tests
               (YOLOv8m learns from the combined wisdom)


In [6]:
 #Verify GPU

# import os
#!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [7]:
# ============================================================
# GCS Authentication (Permanent — Service Account)
# ============================================================

#subprocess — Python run shell commands (like gcloud, gsutil) from inside Python
#threading — Python run a background task simultaneously while training runs
#time — used for time.sleep() (pause) and time.strftime() (print current time)
#os — used for file checks like os.path.exists()

import subprocess, threading, time, os

#below 3 variable definitions — store our paths/IDs
KEY_FILE   = "/content/gcs-key.json"
PROJECT_ID = "solar-cycle-487619-t6"
GCS_BUCKET = "gs://mywaymo-perdataset-2026"

#Defines a reusable function that re-authenticates GCS
def _reauth():
    # tells gcloud "use this service account key for all future GCS operations"
    subprocess.run(["gcloud", "auth", "activate-service-account",
                    "--key-file", KEY_FILE], capture_output=True)
    #— tells gcloud which GCP project to bill/access.
    #capture_output=True — suppresses output so it runs silently in the background without cluttering our notebook
    subprocess.run(["gcloud", "config", "set", "project", PROJECT_ID],
                    capture_output=True)
# calls _reauths() to reauthenticate every 45 minutes by default (tokens expire after ~60 min, so 45 is safe).
#waits 45 minutes (45 × 60 = 2700 seconds)
def start_auth_keepalive(interval_minutes=45):
    #runs forever in a loop
    def _loop():
        while True:
            time.sleep(interval_minutes * 60)
            #silently refreshes the GCS token
            _reauth()
            #shows a timestamp so we know re-auth happened (useful for debugging)
            print(f"🔄 GCS re-auth at {time.strftime('%H:%M:%S')}")
    #threading.Thread(target=_loop) — creates a background thread that runs _loop
    #daemon=True->if the main (colab) program dies, kill this helper (_reauth() thread) too automatically."
    #.start() — start background thread immediately
    threading.Thread(target=_loop, daemon=True).start()
    #print confirms it started
    print(f"🔄 Keepalive started (every {interval_minutes} min)")

# Step 1 — bootstrap: needed interactive login just to grab the key once per session, then we immediately switch to the service account
!gcloud auth login --no-launch-browser #interactive one-time login — opens a URL,paste a code — this auth needed to download the key
#downloads gcs-key.json from my GCS bucket to /content/ on the Colab VM
!gsutil cp gs://mywaymo-perdataset-2026/auth/gcs-key.json /content/gcs-key.json

# Step 2 — immediately switches from the short-lived interactive login to service account (never expires)
_reauth()
start_auth_keepalive()#starts the background thread that re-auths every 45 min for the rest of the session

# Step 3 — verify

#Runs gsutil ls gs://mywaymo-perdataset-2026 to test the connection
#capture_output=True — captures the output instead of printing it directly
#text=True — returns output as a string (not bytes)
#result.returncode == 0 — 0 means success in Linux/shell, anything else is an error
#Prints ✅ if it worked, or the actual error message if it didn't

result = subprocess.run(["gsutil", "ls", GCS_BUCKET], capture_output=True, text=True)
print("✅ GCS authenticated and connected" if result.returncode == 0 else f"❌ {result.stderr}")

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=PaKuoSGckm2O4dCdTsRz8MCJm5YdT6&prompt=consent&token_usage=remote&access_type=offline&code_challenge=p37RoRmfYO98Hn-FhLHUUIw2LaZAwl2ywuQzv7fslnA&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0Aci98E-jZOi5xvWfDLyo_8cZVHgP4vm90K4VVCArx00cogqa9s2ePBoYoPBh2rrajmc9Cg

You are now logged in as [suh2162674@maricopa.edu].
Your current projec

In [8]:
# Install all dependencies for NB7
!pip install rfdetr==1.4.3 -q
!pip install ultralytics -q
!pip install ensemble-boxes -q
print("Installed dependencies✅; restart session")

Installed dependencies✅; restart session


In [1]:
# ============================================================
# Imports
# ============================================================
import os
import csv
import subprocess
import time
import io
from pathlib import Path

import torch
import numpy as np
from rfdetr import RFDETRLarge
from ultralytics import YOLO
from ensemble_boxes import weighted_boxes_fusion
from PIL import Image
print("All imports successful ✅")

All imports successful ✅


In [2]:
# ============================================================
# GLOBAL CONSTANTS — Run this cell first, defines everything
# ============================================================
from pathlib import Path

# Dataset
LOCAL_DATA_DIR      = Path("/content/waymo_yolo")
LOCAL_DATA_DIR.mkdir(exist_ok=True)
CLASS_NAMES         = ["Vehicle", "Pedestrian", "Sign", "Cyclist"]

# Training
LOCAL_OUTPUT_DIR    = "/content/runs/rfdetr_150"
TOTAL_EPOCHS        = 50

# GCS
GCS_BUCKET          = "gs://mywaymo-perdataset-2026"
GCS_CHECKPOINT_PATH = f"{GCS_BUCKET}/checkpoints"
GCS_CSV             = f"{GCS_BUCKET}/models/rfdetr_150_training_history.csv"
GCS_AUTH_KEY        = f"{GCS_BUCKET}/auth/gcs-key.json"

# Local paths
KEY_FILE            = "/content/gcs-key.json"
CSV_PATH            = "/content/training_history.csv"

print("Global constants defined ✅")

Global constants defined ✅


In [3]:
#Restore dataset from GCS

print("Restoring...")
!gsutil -m -q cp -r gs://mywaymo-perdataset-2026/prepared_150/images /content/waymo_yolo/
!gsutil -m -q cp -r gs://mywaymo-perdataset-2026/prepared_150/labels /content/waymo_yolo/
print("Done ✅")

print(f"Train: {len(os.listdir('/content/waymo_yolo/images/train'))}")
print(f"Val: {len(os.listdir('/content/waymo_yolo/images/val'))}")

Restoring...
Done ✅
Train: 23033
Val: 5759


In [ ]:
#Verify both teacher models exist in GCS

!gsutil ls gs://mywaymo-perdataset-2026/models/ | grep -E "rfdetr|yolo26x"

gs://mywaymo-perdataset-2026/models/rfdetr_150_training_history.csv
gs://mywaymo-perdataset-2026/models/waymo_rfdetr_150_best.pth
gs://mywaymo-perdataset-2026/models/waymo_rfdetr_150_last.pth
gs://mywaymo-perdataset-2026/models/waymo_yolo26x_150_best.pt
gs://mywaymo-perdataset-2026/models/waymo_yolo26x_150_last.pt
gs://mywaymo-perdataset-2026/models/waymo_yolo26x_best.pt
gs://mywaymo-perdataset-2026/models/waymo_yolo26x_last.pt
gs://mywaymo-perdataset-2026/models/yolo26x/
gs://mywaymo-perdataset-2026/models/yolo26x_150/


**Download + Load RF-DETR-L teacher:**


In [ ]:
# Download RF-DETR-L teacher from GCS
!gsutil cp gs://mywaymo-perdataset-2026/models/waymo_rfdetr_150_best.pth /content/waymo_rfdetr_150_best.pth
print("RF-DETR-L downloaded ✅")

# Load model
rfdetr_teacher = RFDETRLarge(pretrain_weights='/content/waymo_rfdetr_150_best.pth')
print("RF-DETR-L teacher loaded ✅")

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://mywaymo-perdataset-2026/models/waymo_rfdetr_150_best.pth...
==> NOTE: You are downloading one or more large file(s), which would
run significantly faster if you enabled sliced object downloads. This
feature is enabled by default but requires that compiled crcmod be
installed (see "gsutil help crcmod").

- [1 files][513.9 MiB/513.9 MiB]   14.0 MiB/s                                   
Operation completed over 1 objects/513.9 MiB.                                    


[2026-04-15 22:44:09] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-04-15 22:44:09] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


RF-DETR-L downloaded ✅
[2026-04-15 22:44:09] [INFO] rf-detr - Loading pretrain weights


[2026-04-15 22:44:10] [WARNING] rf-detr - Reinitializing detection head with 3 classes based on pretrained weights, configured for 90.


RF-DETR-L teacher loaded ✅


**Download + Load YOLO26x teacher:**

In [ ]:
# Download YOLO26x teacher from GCS
!gsutil cp gs://mywaymo-perdataset-2026/models/waymo_yolo26x_150_best.pt /content/waymo_yolo26x_150_best.pt
print("YOLO26x downloaded ✅")

# Load model
yolo_teacher = YOLO('/content/waymo_yolo26x_150_best.pt')
print("YOLO26x teacher loaded ✅")

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://mywaymo-perdataset-2026/models/waymo_yolo26x_150_best.pt...
| [1 files][112.8 MiB/112.8 MiB]                                                
Operation completed over 1 objects/112.8 MiB.                                    
YOLO26x downloaded ✅
YOLO26x teacher loaded ✅


**Generate pseudo-labels with WBF**

Stage 1 Config

— Helper: RF-DETR predictions

— Helper: YOLO26x predictions

— Helper: YOLO format converter

— Main: Generate pseudo-labels

In [4]:
# ============================================================
# Stage 1 Configuration
# ============================================================

CONF_THRESHOLD  = 0.35   # ignore boxes below this confidence
IOU_THRESHOLD   = 0.55   # boxes overlapping more than this = same object
RFDETR_WEIGHT   = 0.6    # RF-DETR gets 60% vote (stronger model)
YOLO_WEIGHT     = 0.4    # YOLO26x gets 40% vote
GCS_PSEUDO_PATH = f"{GCS_BUCKET}/pseudo_labels"

# Local folder to save labels before uploading to GCS
LOCAL_PSEUDO_DIR = Path("/content/pseudo_labels")
LOCAL_PSEUDO_DIR.mkdir(exist_ok=True)
(LOCAL_PSEUDO_DIR / "train").mkdir(exist_ok=True)
(LOCAL_PSEUDO_DIR / "val").mkdir(exist_ok=True)

print("Stage 1 config ready ✅")
print(f"Confidence threshold : {CONF_THRESHOLD}")
print(f"IOU threshold        : {IOU_THRESHOLD}")
print(f"RF-DETR weight       : {RFDETR_WEIGHT}")
print(f"YOLO26x weight       : {YOLO_WEIGHT}")

Stage 1 config ready ✅
Confidence threshold : 0.35
IOU threshold        : 0.55
RF-DETR weight       : 0.6
YOLO26x weight       : 0.4


In [ ]:
# ============================================================
# Helper: get_rfdetr_predictions function called once per image inside the main loop
# Returns boxes, scores, labels — all normalized 0-1
# WBF needs 0-1 normalized coordinates
#the main loop calls it for every image (29,700 times total).
# ============================================================
def get_rfdetr_predictions(image_path, img_width, img_height):
    img = Image.open(image_path).convert("RGB")

    # result is a supervision.Detections object
    result = rfdetr_teacher.predict(img)

    boxes, scores, labels = [], [], []

    # result.xyxy = pixel coordinates [[x1,y1,x2,y2], ...]
    # result.confidence = confidence scores [0.94, ...]
    # result.class_id = class ids [0, 1, ...]
    for i in range(len(result.xyxy)):
        x1 = max(0, float(result.xyxy[i][0]) / img_width)
        y1 = max(0, float(result.xyxy[i][1]) / img_height)
        x2 = min(1, float(result.xyxy[i][2]) / img_width)
        y2 = min(1, float(result.xyxy[i][3]) / img_height)

        boxes.append([x1, y1, x2, y2])
        scores.append(float(result.confidence[i]))
        labels.append(int(result.class_id[i]))

    return boxes, scores, labels

**RF-DETR returns a supervision.Detections object**

Before: det['bbox'][0]     ← dict access ❌
After:  result.xyxy[i][0]  ← supervision.Detections attributes ✅

Before: det['score']       ← dict access ❌
After:  result.confidence[i] ← correct attribute ✅

Before: det['label']       ← dict access ❌
After:  result.class_id[i] ← correct attribute ✅

supervision is a Python library made by Roboflow (same company that made RF-DETR) for computer vision tasks.

supervision.Detections is a container that holds all detection results from one image in an organized way:
```
supervision.Detections
│
├── .xyxy          → bounding box coordinates [[x1,y1,x2,y2], ...]
│                    in pixels
│                    Example: [[1503.2, 571.99, 1630.4, 909.63]]
│
├── .confidence    → confidence scores [0.94, 0.87, 0.76, ...]
│                    how sure the model is about each detection
│
├── .class_id      → class numbers [0, 1, 2, ...]
│                    0=Vehicle, 1=Pedestrian, 2=Sign, 3=Cyclist
│
├── .mask          → segmentation masks (None for detection)
│
└── .tracker_id    → tracking IDs (None, not used here)
```
**Simple analogy:**

Imagine RF-DETR looks at an image and finds 3 objects.

Instead of returning messy separate lists,
supervision.Detections packages everything neatly:
```
result.xyxy        = [[box1], [box2], [box3]]
result.confidence  = [0.94,   0.87,   0.76 ]
result.class_id    = [0,      1,      3    ]
                      Vehicle Ped    Cyclist
```

In [ ]:
# ============================================================
# Helper: get_yolo_predictions function called once per image inside the main loop
# Returns boxes, scores, labels — all normalized 0-1
#the main loop calls it for every image (29,700 times total).
# ============================================================
def get_yolo_predictions(image_path, img_width, img_height):
    result = yolo_teacher.predict(image_path, verbose=False)[0]

    boxes, scores, labels = [], [], []
    for box in result.boxes:
        # normalize pixel coords to 0-1
        x1 = max(0, float(box.xyxy[0][0]) / img_width)
        y1 = max(0, float(box.xyxy[0][1]) / img_height)
        x2 = min(1, float(box.xyxy[0][2]) / img_width)
        y2 = min(1, float(box.xyxy[0][3]) / img_height)

        boxes.append([x1, y1, x2, y2])
        scores.append(float(box.conf))
        labels.append(int(box.cls))

    return boxes, scores, labels

In [ ]:
# ============================================================
# Helper: Convert WBF boxes to YOLO label format
#
# YOLO format per line:
# class_id x_center y_center width height  (all normalized 0-1)
# Example: "0 0.5 0.5 0.3 0.4" = Vehicle at image center
# ============================================================
def boxes_to_yolo_format(boxes, scores, labels):
    yolo_lines = []
    for box, score, label in zip(boxes, scores, labels):
        if score < CONF_THRESHOLD:
            continue  # skip low confidence boxes

        x1, y1, x2, y2 = box
        x_center = (x1 + x2) / 2
        y_center = (y1 + y2) / 2
        width    = x2 - x1
        height   = y2 - y1

        yolo_lines.append(
            f"{int(label)} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
        )
    return yolo_lines

**Why Pseudo label**

**Original label:**  1 box for pedestrian at position X
                  (what human annotator drew)


**Pseudo label:**    1 box for pedestrian at position X
                  (what RF-DETR + YOLO26x agreed on)
                  + possibly extra boxes original missed
                  + possibly more accurate coordinates

**Simple analogy:**

Same exam paper (23,033 images)

Different answers written on it:

  Original = Waymo engineer's answers

  Pseudo   = RF-DETR + YOLO26x combined answers
  

The student (YOLOv8m) learns from the pseudo answers — which are potentially better than the original because two strong models agreed on them.

In [ ]:
# ============================================================
# Main— Generate Pseudo-labels
# For each image:
#   1. RF-DETR predicts boxes
#   2. YOLO26x predicts boxes
#   3. WBF merges both → best combined boxes
#   4. Save as YOLO format label file
#   5. Upload to GCS
# ============================================================
def generate_pseudo_labels(split="train"):
    image_dir  = Path(f"/content/waymo_yolo/images/{split}")
    output_dir = LOCAL_PSEUDO_DIR / split
    image_files = sorted(list(image_dir.glob("*.jpg")) +
                         list(image_dir.glob("*.png")))
    total   = len(image_files)
    saved   = 0
    skipped = 0

    print(f"Processing {split}: {total} images...")

    for idx, image_path in enumerate(image_files):

        # get image size
        img = Image.open(image_path)
        img_width, img_height = img.size

        # get predictions from both teachers
        rfdetr_boxes, rfdetr_scores, rfdetr_labels = get_rfdetr_predictions(image_path, img_width, img_height)
        yolo_boxes,   yolo_scores,   yolo_labels   = get_yolo_predictions(image_path, img_width, img_height)

        # save empty label if both models found nothing
        if len(rfdetr_boxes) == 0 and len(yolo_boxes) == 0:
            skipped += 1
            (output_dir / (image_path.stem + ".txt")).write_text("")
            continue

        # WBF — merge both models' predictions
        merged_boxes, merged_scores, merged_labels = weighted_boxes_fusion(
            [rfdetr_boxes,  yolo_boxes],
            [rfdetr_scores, yolo_scores],
            [rfdetr_labels, yolo_labels],
            weights=[RFDETR_WEIGHT, YOLO_WEIGHT],
            iou_thr=IOU_THRESHOLD,
            skip_box_thr=CONF_THRESHOLD
        )

        # convert to YOLO format and save
        yolo_lines = boxes_to_yolo_format(merged_boxes, merged_scores, merged_labels)
        (output_dir / (image_path.stem + ".txt")).write_text("\n".join(yolo_lines))
        saved += 1

        # progress update every 500 images
        if (idx + 1) % 500 == 0:
            print(f"  [{idx+1}/{total}] saved={saved} skipped={skipped}")

    print(f"✅ {split} done: {saved} labels saved, {skipped} empty")

    # upload to GCS
    print(f"Uploading {split} labels to GCS...")
    subprocess.run(["gsutil", "-m", "-q", "cp", "-r",
                    str(output_dir), f"{GCS_PSEUDO_PATH}/"])
    print(f"✅ {split} uploaded to GCS")

# run for both splits
generate_pseudo_labels("train")  # 23,033 images ~2hrs
generate_pseudo_labels("val")    # 5,759 images  ~30min

print("\n🎉 Stage 1 Complete!")
print(f"Pseudo-labels saved to: {GCS_PSEUDO_PATH}/")

Processing train: 23033 images...
  [500/23033] saved=500 skipped=0
  [1000/23033] saved=1000 skipped=0
  [1500/23033] saved=1499 skipped=1
  [2000/23033] saved=1999 skipped=1
  [2500/23033] saved=2499 skipped=1
  [3000/23033] saved=2999 skipped=1
  [3500/23033] saved=3499 skipped=1
  [4000/23033] saved=3999 skipped=1
  [4500/23033] saved=4499 skipped=1
  [5000/23033] saved=4999 skipped=1
  [5500/23033] saved=5499 skipped=1
  [6000/23033] saved=5999 skipped=1
  [6500/23033] saved=6499 skipped=1
  [7000/23033] saved=6999 skipped=1
  [7500/23033] saved=7499 skipped=1
  [8000/23033] saved=7999 skipped=1
  [8500/23033] saved=8499 skipped=1
  [9000/23033] saved=8993 skipped=7
  [9500/23033] saved=9493 skipped=7
  [10000/23033] saved=9993 skipped=7
  [10500/23033] saved=10493 skipped=7
  [11000/23033] saved=10991 skipped=9
  [11500/23033] saved=11491 skipped=9
  [12000/23033] saved=11991 skipped=9
  [12500/23033] saved=12491 skipped=9
  [13000/23033] saved=12991 skipped=9
  [13500/23033] sav

```
[1000/23033] saved=1000 skipped=0
 │    │       │           │
 │    │       │           └── 0 images where BOTH models found nothing
 │    │       │                (no objects detected at all)
 │    │       │
 │    │       └── 1000 label files successfully saved
 │    │            (had at least 1 detection after WBF)
 │    │
 │    └── total images to process (23,033 train images)
 │
 └── images processed so far

[1500/23033] saved=1499 skipped=1
                          │
                          └── 1 image where both RF-DETR AND YOLO26x
                              found zero objects
                              (empty label file saved — normal!)
```

#Final Summary:
Train: 23,020 labels saved  13 empty  (0.056%) ✅

Val:    5,751 labels saved   8 empty  (0.139%) ✅

Total: 28,771 pseudo-labels generated from ensemble

#Quality check — very healthy numbers:

Only 21 empty images out of 28,792 total (0.07%)

Means both teachers found objects in 99.93% of images ✅



In [ ]:
#verify pseudo-labels uploaded correctly to GCS:

#!gsutil ls gs://mywaymo-perdataset-2026/pseudo_labels/
#!gsutil ls gs://mywaymo-perdataset-2026/pseudo_labels/train/ | wc -l
#!gsutil ls gs://mywaymo-perdataset-2026/pseudo_labels/val/ | wc -l

#avoid count the folder line;Use grep to filter only .txt files
!gsutil ls gs://mywaymo-perdataset-2026/pseudo_labels/train/ | grep ".txt" | wc -l
!gsutil ls gs://mywaymo-perdataset-2026/pseudo_labels/val/ | grep ".txt" | wc -l

23033
5759


In [5]:
# ============================================================
# Restore pseudo-labels from GCS
# ============================================================

# Step 1 — create local directories first
os.makedirs("/content/pseudo_labels/train", exist_ok=True)
os.makedirs("/content/pseudo_labels/val", exist_ok=True)
print("Directories created ✅")

# Step 2 — download from GCS
print("Downloading train pseudo-labels...")
subprocess.run([
    "gsutil", "-m", "-q", "cp",
    "gs://mywaymo-perdataset-2026/pseudo_labels/train/*.txt",
    "/content/pseudo_labels/train/"
])

print("Downloading val pseudo-labels...")
subprocess.run([
    "gsutil", "-m", "-q", "cp",
    "gs://mywaymo-perdataset-2026/pseudo_labels/val/*.txt",
    "/content/pseudo_labels/val/"
])

# Step 3 — verify
train = len(os.listdir('/content/pseudo_labels/train'))
val   = len(os.listdir('/content/pseudo_labels/val'))
print(f"Train pseudo-labels: {train}")
print(f"Val pseudo-labels:   {val}")
print("✅ Pseudo-labels restored")

Directories created ✅
Train pseudo-labels: 23033
Val pseudo-labels:   5759
✅ Pseudo-labels restored


In [6]:
# ============================================================
# Copy restored pseudo-labels into waymo_yolo folder structure
# ============================================================
import shutil

os.makedirs("/content/waymo_yolo/pseudo_labels/train", exist_ok=True)
os.makedirs("/content/waymo_yolo/pseudo_labels/val", exist_ok=True)

shutil.copytree("/content/pseudo_labels/train",
                "/content/waymo_yolo/pseudo_labels/train",
                dirs_exist_ok=True)
shutil.copytree("/content/pseudo_labels/val",
                "/content/waymo_yolo/pseudo_labels/val",
                dirs_exist_ok=True)

# Verify
train = len(os.listdir("/content/waymo_yolo/pseudo_labels/train"))
val   = len(os.listdir("/content/waymo_yolo/pseudo_labels/val"))
print(f"Train: {train}")
print(f"Val:   {val}")
print("✅ Pseudo-labels ready in waymo_yolo")

Train: 23033
Val:   5759
✅ Pseudo-labels ready in waymo_yolo



```
Before swap:
waymo_yolo/
├── labels/          ← original Waymo labels (YOLO reads this by default)
└── pseudo_labels/   ← ensemble labels (YOLO ignores this)

After swap:
waymo_yolo/
├── labels/          ← ensemble pseudo-labels (YOLO reads this now) ✅
└── labels_original/ ← original Waymo labels (safe backup)
```

In [7]:
#Swap labels folder to tell YOLO to use ensemble labels instead of original Waymo labels.

# Check current state
folders = os.listdir('/content/waymo_yolo')
print(f"Current folders: {folders}")

# Only swap if not already done
if 'labels_original' not in folders:
    os.rename("/content/waymo_yolo/labels",
              "/content/waymo_yolo/labels_original")
    os.rename("/content/waymo_yolo/pseudo_labels",
              "/content/waymo_yolo/labels")
    print("Swap done ✅")
else:
    print("Already swapped ✅ skipping")

Current folders: ['images', 'labels', 'pseudo_labels']
Swap done ✅


In [8]:
# ============================================================
#Create data_pseudo.yaml — that has pseudo-labels
#  — standard YOLO format
# labels/ folder now contains pseudo-labels
# ============================================================
yaml_content = """path: /content/waymo_yolo
train: images/train
val: images/val

nc: 4
names:
  - Vehicle
  - Pedestrian
  - Sign
  - Cyclist
"""

with open("/content/waymo_yolo/data_pseudo.yaml", "w") as f:
    f.write(yaml_content)

print("data_pseudo.yaml fixed ✅")
print(open("/content/waymo_yolo/data_pseudo.yaml").read())

data_pseudo.yaml fixed ✅
path: /content/waymo_yolo
train: images/train
val: images/val

nc: 4
names:
  - Vehicle
  - Pedestrian
  - Sign
  - Cyclist



**Folder structure is now:**

```
waymo_yolo/
├── labels_original/   ← original Waymo ground truth (safe backup)
│   ├── train/
│   └── val/
├── labels/            ← pseudo-labels from ensemble (YOLO reads this)
│   ├── train/
│   └── val/
├── images/
│   ├── train/
│   └── val/
└── data.yaml
```

In [9]:
#clear cache
import os

# Delete old cache — forces YOLO to rescan pseudo-labels
for cache in ['/content/waymo_yolo/labels/train.cache',
              '/content/waymo_yolo/labels/val.cache']:
    if os.path.exists(cache):
        os.remove(cache)
        print(f"Deleted: {cache}")
    else:
        print(f"Not found (ok): {cache}")

print("Cache cleared ✅")

Not found (ok): /content/waymo_yolo/labels/train.cache
Not found (ok): /content/waymo_yolo/labels/val.cache
Cache cleared ✅


In [ ]:
# ============================================================
#Stage 2 — Train YOLOv8m Student Model
# Student learns from ensemble pseudo-labels
# GCS callback saves checkpoint after every epoch
# Resume from previous run's last epoch (RF-DETR pattern)
# Automatically finds last checkpoint and resumes
# ============================================================


GCS_STUDENT_PATH = f"{GCS_BUCKET}/models/student"

# ---- Step 1: Find last saved epoch from GCS ----
result = subprocess.run(
    ["gsutil", "ls", f"{GCS_STUDENT_PATH}/"],
    capture_output=True, text=True
)
checkpoints = [
    line for line in result.stdout.strip().split('\n')
    if 'epoch_' in line
]
checkpoints.sort(
    key=lambda x: int(x.split('epoch_')[1].replace('.pt', ''))
)

if checkpoints:
    last_checkpoint_gcs = checkpoints[-1]
    last_epoch = int(last_checkpoint_gcs.split('epoch_')[1].replace('.pt', ''))
    print(f"Found {len(checkpoints)} checkpoints")
    print(f"Resuming from epoch: {last_epoch}")
else:
    last_checkpoint_gcs = None
    last_epoch = 0
    print("No checkpoints found — starting fresh")

# ---- Step 2: Download last checkpoint ----
if last_epoch > 0:
    local_checkpoint = f"/content/yolov8m_distilled_epoch_{last_epoch}.pt"
    subprocess.run(["gsutil", "cp", last_checkpoint_gcs, local_checkpoint])
    print(f"Checkpoint downloaded ✅")
    print(f"Exists: {os.path.exists(local_checkpoint)}")
    print(f"Size: {os.path.getsize(local_checkpoint)/1e6:.1f} MB")

# ---- Step 3: GCS callback ----
def save_student_to_gcs(trainer):
    epoch = trainer.epoch + last_epoch + 1  # real epoch number

    subprocess.run(["gcloud", "auth", "activate-service-account",
                    "--key-file", KEY_FILE], capture_output=True)

    #trainer.save_dir is a variable that YOLO sets automatically,YOLO internally sets this based on
    # project + whatever name YOLO chose
    ## Let YOLO tell us where it saved
    last_weights = os.path.join(str(trainer.save_dir), "weights/last.pt")
    if os.path.exists(last_weights):
        subprocess.run(["gsutil", "-q", "cp", last_weights,
                        f"{GCS_STUDENT_PATH}/yolov8m_distilled_epoch_{epoch}.pt"],
                       capture_output=True, timeout=60)
        print(f"✅ Epoch {epoch} checkpoint saved to GCS")

    best_weights  = os.path.join(str(trainer.save_dir), "weights/best.pt")
    if os.path.exists(best_weights):
        subprocess.run(["gsutil", "-q", "cp", best_weights,
                        f"{GCS_STUDENT_PATH}/yolov8m_distilled_best.pt"],
                       capture_output=True, timeout=60)
        print(f"✅ Epoch {epoch} best saved to GCS")

# ---- Step 4: Load model ----
if last_epoch > 0:
    student = YOLO(local_checkpoint)
    print(f"Student loaded from epoch {last_epoch} ✅")
else:
    student = YOLO("yolov8m.pt")
    print("Student loaded fresh ✅")

# ---- Step 5: Attach callback ----
student.add_callback("on_fit_epoch_end", save_student_to_gcs)
print("GCS callback attached ✅")

# ---- Step 6: Train remaining epochs ----
remaining_epochs = 50 - last_epoch
print(f"Remaining epochs: {remaining_epochs}")

student.train(
    data= "/content/waymo_yolo/data_pseudo.yaml",
    epochs=remaining_epochs,
    imgsz=640,
    batch=8,
    workers=0,
    device=0,
    project="/content/runs",
    name="yolov8m_distilled",
    save=True,
    patience=40,
    lr0=0.01,
    cos_lr=True,
)

print("✅ Student training complete!")

Found 34 checkpoints
Resuming from epoch: 34
Checkpoint downloaded ✅
Exists: True
Size: 207.4 MB
Student loaded from epoch 34 ✅
GCS callback attached ✅
Remaining epochs: 16
Ultralytics 8.4.38 🚀 Python-3.12.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/waymo_yolo/data_pseudo.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=16, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=

In [ ]:
# Check all saved checkpoints
!gsutil ls gs://mywaymo-perdataset-2026/models/student/ | sort -t_ -k4 -n


gs://mywaymo-perdataset-2026/models/student/yolov8m_distilled_best.pt
gs://mywaymo-perdataset-2026/models/student/yolov8m_distilled_epoch_1.pt
gs://mywaymo-perdataset-2026/models/student/yolov8m_distilled_epoch_2.pt
gs://mywaymo-perdataset-2026/models/student/yolov8m_distilled_epoch_3.pt
gs://mywaymo-perdataset-2026/models/student/yolov8m_distilled_epoch_4.pt
gs://mywaymo-perdataset-2026/models/student/yolov8m_distilled_epoch_5.pt
gs://mywaymo-perdataset-2026/models/student/yolov8m_distilled_epoch_6.pt
gs://mywaymo-perdataset-2026/models/student/yolov8m_distilled_epoch_7.pt
gs://mywaymo-perdataset-2026/models/student/yolov8m_distilled_epoch_8.pt
gs://mywaymo-perdataset-2026/models/student/yolov8m_distilled_epoch_9.pt
gs://mywaymo-perdataset-2026/models/student/yolov8m_distilled_epoch_10.pt
gs://mywaymo-perdataset-2026/models/student/yolov8m_distilled_epoch_11.pt
gs://mywaymo-perdataset-2026/models/student/yolov8m_distilled_epoch_12.pt
gs://mywaymo-perdataset-2026/models/student/yolov8m

In [10]:
# Download and evaluate best model
!gsutil cp gs://mywaymo-perdataset-2026/models/student/yolov8m_distilled_best.pt \
           /content/yolov8m_distilled_best.pt

from ultralytics import YOLO
student_best = YOLO('/content/yolov8m_distilled_best.pt')

results = student_best.val(
    data="/content/waymo_yolo/data_pseudo.yaml",
    device=0,
    workers=0,
)
print(f"mAP@0.5:     {results.box.map50:.3f}")
print(f"mAP@0.5:0.95:{results.box.map:.3f}")

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://mywaymo-perdataset-2026/models/student/yolov8m_distilled_best.pt...
/ [1 files][ 49.6 MiB/ 49.6 MiB]                                                
Operation completed over 1 objects/49.6 MiB.                                     
Ultralytics 8.4.38 🚀 Python-3.12.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Model summary (fused): 93 layers, 25,842,076 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3050.4±951.5 MB/s, size: 210.9 KB)
val: Scanning /content/waymo_yolo/labels/val... 5759 images, 13 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5759/5759 1.4Kit/s 4.0s
val: New cache created: /content/waymo_yolo/labels/val.cache
                 Class     Images  Instances

In [11]:
#Save the model to GCS
!gsutil cp /content/yolov8m_distilled_best.pt \
           gs://mywaymo-perdataset-2026/models/waymo_yolov8m_distilled_best.pt
print("✅ Best student model saved to GCS!")

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file:///content/yolov8m_distilled_best.pt [Content-Type=application/vnd.snesdev-page-table]...
-
Operation completed over 1 objects/49.6 MiB.                                     
✅ Best student model saved to GCS!


## Lessons Learned — Ensemble Distillation

### Stage 1 — Pseudo-label Generation
- RF-DETR returns `supervision.Detections` object — not a dict. Use `result.xyxy[i]`, `result.confidence[i]`, `result.class_id[i]`
- RF-DETR `predict()` requires PIL Image — not a file path. Use `Image.open(path).convert('RGB')`
- WBF needs normalized 0-1 coordinates — always divide by image width/height before passing to WBF
- Stream gsutil to stdout (`-`) fails for large .pth files — download locally instead
- Use `os.makedirs(exist_ok=True)` before gsutil cp to avoid `CommandException: Destination must be a directory`

### Stage 2 — Student Training
- YOLO ignores custom yaml fields (`train_labels`, `val_labels`) — always uses `labels/` folder
- Correct approach: rename `pseudo_labels/` → `labels/` and backup original as `labels_original/`
- Add safety check before swap: `if 'labels_original' not in os.listdir(...)` to prevent double swap
- Use `trainer.save_dir` dynamically in callback — never hardcode the run folder name
- `caffeinate` command on Mac prevents sleep during long training runs
- `workers=0` required on Colab to prevent dataloader deadlock

### Results
- **Distillation beat all previous models** — mAP@0.5 = 0.866 vs RF-DETR-L's 0.826
- **10x faster than RF-DETR** — 4.8ms vs ~50ms inference
- **+13% improvement over original YOLOv8m** — 0.866 vs 0.736
- Student model saved: `gs://mywaymo-perdataset-2026/models/waymo_yolov8m_distilled_best.pt`

### Final Results
| Class | Precision | Recall | mAP@0.5 | mAP@0.5:0.95 |
|-------|-----------|--------|---------|-------------|
| Vehicle | 0.942 | 0.849 | 0.908 | 0.719 |
| Pedestrian | 0.912 | 0.797 | 0.880 | 0.627 |
| Cyclist | 0.871 | 0.745 | 0.808 | 0.539 |
| **All** | **0.908** | **0.797** | **0.866** | **0.629** |
